# Amazon baseline improvements for Person A

Select by mean average precision in three product-disjoint development folds. Selection uses reviews before the time boundary and excludes the threshold reviewers and all final holdout products. Each blend averages the component probabilities. Choose each final cutoff on reviewer fold 0. The two evaluations use separate fits. The forward-time backtest scores later reviews on development products. The final bundle uses all dates in training folds 1–4 and scores the previously examined product holdout. All positive labels use the audited Amazon combined rule.

In [1]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if root.name == 'notebooks_v2':
    root = root.parent
if (root / "Person A").is_dir():
    root = root / "Person A"
sys.path.insert(0, str(root))
from review_pipeline.person_a_inputs import shared_project
dependencies = shared_project(root) / ".v2-deps"
if dependencies.is_dir():
    sys.path.insert(0, str(dependencies))
from review_pipeline.person_a_prospective import run_experiment, score_reviews
import pandas as pd

In [2]:
directory, metrics = run_experiment(root)
print(directory)
display(metrics[['cohort','model','n','roc_auc','pr_auc','ap','precision','recall','f1','tn','fp','fn','tp']])
display(pd.read_csv(directory / 'selection.csv').groupby('method').ap.mean().sort_values(ascending=False))
display(pd.read_csv(directory / 'paired_intervals.csv'))

Completed development product fold 1/3


Completed development product fold 2/3


Completed development product fold 3/3


Selected history_tree on earlier development products.


C:\Users\HP\OneDrive\Desktop\pe\results\person_a\prospective_20261121T180956177896Z


,cohort,model,n,roc_auc,pr_auc,ap,precision,recall,f1,tn,fp,fn,tp
0,forward_time,rating,46748,0.909501,0.945429,0.913394,0.943436,0.888872,0.915342,18228,1443,3009,24068
1,forward_time,history_tree,46748,0.952819,0.965003,0.965004,0.897511,0.906858,0.902160,16867,2804,2522,24555
2,product_holdout,rating,74192,0.827241,0.732597,0.628974,0.790703,0.683082,0.732963,56520,2706,4743,10223
3,product_holdout,history_tree,74192,0.887807,0.821766,0.821770,0.884827,0.666310,0.760177,57928,1298,4994,9972
4,product_holdout,week1_retrospective,74192,0.849316,0.547700,0.547736,0.698021,0.690699,0.694341,54754,4472,4629,10337


method
history_tree     0.741981
history_blend    0.733646
local_blend      0.647542
word_model       0.631628
local_tree       0.628827
rating           0.527248
Name: ap, dtype: float64

,metric,delta,lower_95,upper_95,product_clusters,draws
0,average_precision,0.274034,0.003224,0.411968,673,400
1,f1,0.065836,-0.006209,0.155721,673,400


### Using the frozen model

Use `score_reviews(directory, source_frame)` with the original feature notebook’s per-review columns, `review_text`, `prod_id`, `rating` and `date`. Supply one platform at a time and include its available earlier product reviews in the same frame. Returned scores and flags align to the input rows. An empty product history is supported. Score only the rows of interest after constructing their history. The older Week 1 model and C’s original files remain available for their assigned comparison.